# Inference: depAPI-prompt-tuning-codegen2b (HF Hub)

Load trained soft prompt từ `HuyTran1301/depAPI-prompt-tuning-codegen2b` và chạy forget-quality evaluation trên tập `outdated_y+_FINAL.json`.

In [ ]:
# Cell 1: Clone hoặc pull code mới nhất
import subprocess, os

REPO_URL = "https://github.com/trhuyyy13/prompt-tuning-api-migration"
REPO_DIR = "/kaggle/working/repo"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    result = subprocess.run(["git", "-C", REPO_DIR, "pull"], capture_output=True, text=True)
    print(result.stdout or result.stderr)
else:
    result = subprocess.run(["git", "clone", REPO_URL, REPO_DIR], capture_output=True, text=True)
    print(result.stdout or result.stderr)

print("[*] repo at:", REPO_DIR)

In [ ]:
# Cell 2: Cài dependencies
!pip install -q -r /kaggle/working/repo/prompt_tuning_deepseek/requirements.txt

In [ ]:
# Cell 3: HF token (cần để snapshot_download không bị rate-limit)
try:
    from kaggle_secrets import UserSecretsClient
    import os
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("[*] HF_TOKEN loaded")
except Exception as e:
    print(f"[warn] Could not load HF_TOKEN: {e} -- continuing without token")

In [ ]:
# Cell 4: Config -- chỉnh tại đây nếu cần
HF_CHECKPOINT  = "HuyTran1301/depAPI-prompt-tuning-codegen2b"
DATA_FILE      = "/kaggle/working/repo/data_raw/outdated_y+_FINAL.json"
OUTPUT_DIR     = "/kaggle/working/outputs/eval_hf_codegen2b"
MAX_NEW_TOKENS = 128
BATCH_SIZE     = 8     # tăng lên nếu GPU còn dư VRAM (T4: 8 là safe với codegen-2B)
LIMIT          = None  # None = toàn bộ 9087 samples; dùng số nguyên để test nhanh

print(f"checkpoint  : {HF_CHECKPOINT}")
print(f"data        : {DATA_FILE}")
print(f"output      : {OUTPUT_DIR}")
print(f"batch_size  : {BATCH_SIZE}")
print(f"limit       : {LIMIT if LIMIT else 'all'}")

In [ ]:
# Cell 5: Inference -- model_name_or_path dibaca otomatis dari prompt_config.json
import subprocess, sys

cmd = [
    sys.executable,
    "/kaggle/working/repo/prompt_tuning_deepseek/test_forget_quality.py",
    "--checkpoint_dir", HF_CHECKPOINT,
    "--data_file",      DATA_FILE,
    "--output_dir",     OUTPUT_DIR,
    "--max_new_tokens", str(MAX_NEW_TOKENS),
    "--batch_size",     str(BATCH_SIZE),
]
if LIMIT is not None:
    cmd += ["--limit", str(LIMIT)]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd="/kaggle/working/repo/prompt_tuning_deepseek")
if result.returncode != 0:
    raise RuntimeError(f"Script exited with code {result.returncode}")

In [ ]:
# Cell 6: Xem kết quả metrics
import json, os

metrics_path = os.path.join(OUTPUT_DIR, "forget_quality_metrics.json")
with open(metrics_path) as f:
    metrics = json.load(f)

print(json.dumps(metrics, indent=2))

In [ ]:
# Cell 7 (optional): Xem một số predictions cụ thể
import json, os

details_path = os.path.join(OUTPUT_DIR, "forget_quality_details.json")
with open(details_path) as f:
    details = json.load(f)

for i, d in enumerate(details[:5]):
    print(f"--- [{i}] type={d['type']} ---")
    print(f"  input   : {d['probing_input'][:120]}")
    print(f"  target  : {d['target'][:120]}")
    print(f"  predict : {d['predict'][:120]}")
    print()